In [97]:
import pandas as pd
from pathlib import Path
import numpy as np

In [98]:
ROOT_DIR = Path.cwd()
orders = ROOT_DIR/'data'/'orders_task.csv'
customers = ROOT_DIR/'data'/'customers_task.csv'
order_items = ROOT_DIR/'data'/'order_items_task.csv'
products = ROOT_DIR/'data'/'products_task.csv'

df1 = pd.read_csv(orders)
df2 = pd.read_csv(customers)
df3 = pd.read_csv(order_items)
df4 = pd.read_csv(products)

df5 = pd.merge(df1, df2, how='left', on='customer_id')
df6 = pd.merge(df5, df3, how='left', on='order_id')
df7 = pd.merge(df6, df4, how='left', on='product_id')
df7

df = df7[df7['status'] == 'Completed']
df

,order_id,customer_id,order_date,status,name,country,city,segment,signup_date,item_id,product_id,quantity,product_name,category,price,cost
2,O0002,C022,2024-02-08,Completed,Greta W.,Nepal,Pokhara,Corporate,2023-05-03,L0003,P08,6,Bookshelf,Furniture,313.58,175.71
3,O0002,C022,2024-02-08,Completed,Greta W.,Nepal,Pokhara,Corporate,2023-05-03,L0004,P05,10,Power Bank,Electronics,297.41,172.35
4,O0003,C013,2024-09-15,Completed,Sofia N.,Australia,Melbourne,Home Office,2023-04-23,L0005,P09,3,Filing Cabinet,Furniture,179.75,143.54
5,O0004,C028,2024-03-09,Completed,Zoe C.,Germany,Munich,Consumer,2023-01-28,L0006,P18,5,Lunch Box,Kitchen,103.66,58.49
6,O0004,C028,2024-03-09,Completed,Zoe C.,Germany,Munich,Consumer,2023-01-28,L0007,P06,2,Office Chair,Furniture,359.18,225.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
902,O0299,C002,2024-01-26,Completed,Rajesh C.,India,Delhi,Consumer,2023-12-04,L0903,P04,7,Webcam,Electronics,107.78,69.26
903,O0299,C002,2024-01-26,Completed,Rajesh C.,India,Delhi,Consumer,2023-12-04,L0904,P23,3,Jump Rope,Sports,94.74,59.41
904,O0299,C002,2024-01-26,Completed,Rajesh C.,India,Delhi,Consumer,2023-12-04,L0905,P16,2,Coffee Mug,Kitchen,136.76,103.65
905,O0300,C016,2024-03-26,Completed,Ravi Q.,USA,Austin,Home Office,2023-06-06,L0906,P13,2,Sticky Notes,Stationery,18.47,10.51


Report total realized revenue, total profit, and the overall profit margin.

In [99]:
df['revenue'] = df['quantity'] * df['price']
df['profit'] = df['revenue'] - (df['quantity'] * df['cost'])
rev = df['revenue'].sum()
prof = df['profit'].sum()

print(f'Total Revenue: {rev}')
print(f'Total Profit: {prof}')
print(f'the overall profit margin is: {prof/rev * 100}%')

Total Revenue: 666213.5700000001
Total Profit: 246609.44999999998
the overall profit margin is: 37.01657563054441%


Rank the product categories by revenue and by profit margin.

In [100]:
top_cat = df.groupby('category').agg(
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum')
).sort_values(by='total_revenue', ascending=False).head(10)

top_cat

,total_revenue,total_profit
category,,
Electronics,278762.65,108601.78
Furniture,232074.07,85066.68
Sports,73040.38,23091.51
Kitchen,70333.29,24598.32
Stationery,12003.18,5251.16


List the top 10 products by revenue, with units sold and profit margin for each

In [101]:
def prof_m(df):
    return df['profit'] / df['revenue']


top_prod = df.groupby('product_name').agg(
    total_revenue=('revenue', 'sum'),
    profit_marg=('profit_marg', 'mean'),
    units_sold = ('quantity', 'sum')
).sort_values(by='total_revenue', ascending=False)
top_prod

KeyError: "Label(s) ['profit_marg'] do not exist"

Compute revenue and average order value by customer segment.

In [ ]:
segment = df.groupby('segment').agg(
    total=('revenue', 'sum'),
    avg=('quantity', 'mean')
).round(2)
segment

,total,avg
segment,,
Consumer,217227.20,5.78
Corporate,57530.97,6.53
Home Office,391455.40,5.56


Compute revenue and average order value by country.

In [ ]:
count = df.groupby('country').agg(
    total=('revenue', 'sum'),
    avg=('revenue', 'mean')
)
count

,total,avg
country,,
Australia,157176.20,863.605495
Germany,119305.18,817.158767
India,53320.48,903.736949
Nepal,126236.89,1026.316179
UK,23309.74,971.239167
USA,186865.08,907.112039


Produce a monthly revenue series and its month-over-month growth.

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

monthly = df.groupby(df['order_date'].dt.to_period('M')).agg(
    total_revenue=('revenue', 'sum')
)
monthly['mom'] = monthly['total_revenue'].pct_change() * 100
monthly

,total_revenue,mom
order_date,,
2024-01,50473.08,NaN
2024-02,54394.34,7.769013
2024-03,45684.58,-16.012254
2024-04,43080.06,-5.701092
2024-05,61294.54,42.280535
2024-06,50156.36,-18.171570
2024-07,47928.45,-4.441929
2024-08,57580.97,20.139437
2024-09,41059.77,-28.692118


Determine the share of total revenue coming from the top 10% of customers.

In [ ]:
tc = df.groupby('customer_id').agg(
    total=('revenue', 'sum')
).sort_values(by='total', ascending=False)
length = len(tc)
top = tc.head(int(length * 0.1))
print(f'total share from top 10% is: {top["total"].sum() / tc["total"].sum() * 100}')

total share from top 10% is: 15.462487502318515


Identify the products that appear in the greatest number of distinct orders.

In [ ]:
top_prod = df.groupby('product_name').agg(
    total = ('order_id', 'count')
)
top_prod = top_prod.sort_values(by='total', ascending=False)
top_prod

,total
product_name,
Lunch Box,42
Desk Lamp,41
Power Bank,40
Water Bottle,38
Jump Rope,37
Bluetooth Speaker,37
Pen Set,35
USB-C Hub,33
Marker Pack,33


Compute the average number of line-items per order and the average number of
units per order.

In [ ]:
orders = df.groupby('order_id').agg(
    avg_items=('quantity', 'mean')
).round(2)
orders

,avg_items
order_id,
O0002,8.00
O0003,3.00
O0004,3.50
O0005,1.00
O0006,5.33
...,...
O0296,8.00
O0297,10.00
O0298,4.50


Compute the non-completion (Cancelled or Returned) rate for each segment.

In [ ]:
not_completed = df7[df7['status'] != 'Completed']
not_completed_segment = not_completed.groupby('segment').agg(
    total=('order_id', 'count')
)
completed = df7[df7['status'] == 'Completed']
completed_segment = completed.groupby('segment').agg(
    total = ('order_id', 'count')
)
not_completed_segment['total'] = (not_completed_segment['total'] / completed_segment['total']).round(2)
not_completed_segment

,total
segment,
Consumer,0.27
Corporate,0.34
Home Office,0.18


Identify the customers who have never placed an order and the products that have
never been sold.

In [ ]:
not_order = df4[~df4['product_id'].isin(df3['product_id'])]
not_order

,product_id,product_name,category,price,cost
24,P25,Resistance Bands,Sports,139.57,110.99


,product_id,product_name,category,price,cost
24,P25,Resistance Bands,Sports,139.57,110.99


Determine the top-selling product by revenue in each country.

In [ ]:
tp = df.groupby(['country', 'product_name'], as_index=False)['revenue'].sum()
tpc = tp.sort_values(by='revenue', ascending=False).groupby('country').first()
tpc

,product_name,revenue
country,,
Australia,Wireless Mouse,25234.89
Germany,Bluetooth Speaker,23143.93
India,Office Chair,7901.96
Nepal,Bluetooth Speaker,18436.69
UK,USB-C Hub,4567.90
USA,Office Chair,25142.60


For each country, identify the customer contributing the largest share of that
country's revenue.

In [ ]:
top_cust = df.groupby(['country', 'name'], as_index=False)['revenue'].sum()
top_cust = top_cust.groupby('country').first()
top_cust

,name,revenue
country,,
Australia,Aisha B.,14742.76
Germany,Gita H.,22419.63
India,Chen X.,13145.19
Nepal,Arjun L.,12425.36
UK,Priya M.,4748.84
USA,Emma H.,21542.84


Rank the categories by profit and report each category's cumulative share of total
profit.

In [ ]:
cat = df.groupby('category').agg(
    total=('profit', 'sum')
)
total = cat['total'].sum()
cat['share_total'] = (cat['total'] / total * 100).round(2)
cat

,total,share_total
category,,
Electronics,108601.78,44.04
Furniture,85066.68,34.49
Kitchen,24598.32,9.97
Sports,23091.51,9.36
Stationery,5251.16,2.13


Identify the single highest-revenue product and the single highest-margin product.

In [109]:
highest = df.groupby('product_name').agg(
    total = ('revenue', 'sum'),
    highest_margin = ('profit_marg', 'max')
)
big_rev = highest.sort_values(by='total', ascending=False).head(1)
big_marg = highest.sort_values(by='highest_margin', ascending=False).head(1)
print(f'highest product revenue is: {big_rev}, highest margin product is: {big_marg}')

highest product revenue is:                       total  highest_margin
product_name                               
Bluetooth Speaker  92183.45        0.270248, highest margin product is:                 total  highest_margin
product_name                         
USB-C Hub     47559.9        0.479233
